# AlphaGenome Introductory Practical Session

Welcome to the AlphaGenome practical session! This notebook will guide you through the basics of using AlphaGenome for genomic data analysis.  
This is not a quick start-tutorial or meant to provide introduction to all of AlphaGenome's capabilities.  It is meant more as a playground and straw man to encourage group discussion.

For a more comprehensive walkthrough, please check out the [AlphaGenome Quick Start Collab](https://colab.research.google.com/github/google-deepmind/alphagenome/blob/main/colabs/quick_start.ipynb) or the more comprehensive [AlphaGenome Documentation](https://www.alphagenomedocs.com/index.html) with more extensive code examples.

## Key Takeaways: 
- The primary input to AlphaGenome is `genomic sequence` (...but also accepts: `genomic coordinates`, `context windows` such as gene models).
- AlphaGenome predicts the molecular consequences of DNA sequences; **not trait or disease causality**.
- It outputs functional genomics (gene regulatory) signals at base resolution, in the **limited context** (tissues/cells) on which it has been trained.
- _It is useful_ when a signal is **sequence-driven** and **generalizable** or when you care about **relative prioritization** (e.g., ranking variants).
- _It is not useful_ when biology is strongly **context-specific** and mismatched to the training.
- Don't treat outputs as **truth**; compare variants under the same context (relative ranking) or look for **consistent signals** (or _lack thereof_) across multiple predicted assays.

In [ ]:
# install requirements
!pip install -r requirements.txt

## 1. First Steps -> AlphaGenome API KEY

To use the model, you will need an [AlphaGenome API Key](https://deepmind.google.com/science/alphagenome). The `API key` needs to be provided to the model at load time.  If you have not yet done so, please follow the link to generate your `API Key`.  

Run the following cell to generate a user prompt and supply your key to the notebook.

In [ ]:
# prompt user for API KEY
from getpass import getpass

api_key = getpass("Enter your AlphaGenome API key: ")

## 2. Import `alphagenome` Python package

AlphaGenome provides an API client.  It is installed in this Jupyter environment so no need to do anything here, but for future reference, it is available in the `PyPI` repository, so can be installed with `pip` or according to the recommended method for a managed build environment (e.g., `conda`, `poetry`).

Run the following cell to import alphagenome client tools, pandas, and a couple visualization generators into this notebook.

In [ ]:
# Install and Import libraries
from alphagenome.data.genome import Variant
from alphagenome.data import gene_annotation, genome, track_data, transcript
from alphagenome.interpretation import ism
from alphagenome.models import dna_client, variant_scorers
from alphagenome.visualization import plot_components
import matplotlib.pyplot as plt
import pandas as pd

# for displaying interactive tables
import itables

itables.init_notebook_mode(all_interactive=True)

## 3. Before we start - load model and auxiliary objects

### 3A. Initial Client / Load Moadel

To leverage the API client and "initialize" the model, you will need to provide your API Key.

In [ ]:
# initalize the AlphaGenome API client w/the API KEY
dna_model = dna_client.create(api_key)

### 3B. Metadata objects for humans

For specific details on AlphaGenoeme outputs, please check out the [AlphaGenome Quick Start](https://colab.research.google.com/github/google-deepmind/alphagenome/blob/main/colabs/quick_start.ipynb).

AlphaGenome predicts molecular consequences by generating multiple assay-specific signal tracks, each conditioned on a particular biosample (cell/tissue) context.

Allowable output types are enumerated by `dna_client.OutputType`.  
Generated tracks can be filtered by tissue and cell lines by leveraging a limited set of terms from the UBERON and Cell Line (CL) ontologies.  

> **Important** Tracks can only be filtered for tissues/cells used to train the model (a selection of mainly ENCODE and FANTOM5 tracks). Not all predicted outputs are available for all biosample types. 

Allowable terms are accesible via the instantiated model's `output_metadata` for the organism of interest.

In [ ]:
# get human track metadata
output_metadata = dna_model.output_metadata(
    organism=dna_client.Organism.HOMO_SAPIENS
).concatenate()

# how many tracks per output type
human_tracks = (
    output_metadata
    .groupby('output_type')
    .size()
    .rename('# Tracks')
)

human_tracks


In [ ]:
# browseable table of all metadata annotations
# if your notebook is slow, reduce maxBytes to 8KB to truncate this table
itables.options.maxBytes = "1MB"
output_metadata


### 3B. Gene Annotations (from GENCODE)

Read in the GENCODE GTF file, filter for protein-coding genes and highly supported transcripts, then instantiate an extractor for identifying transcripts in a region of interest.

In [ ]:
# Load gene annotations (from GENCODE).
gtf = pd.read_feather(
    "https://storage.googleapis.com/alphagenome/reference/gencode/"
    "hg38/gencode.v46.annotation.gtf.gz.feather"
)

# Filter to protein-coding genes and highly supported transcripts.
gtf_transcript = gene_annotation.filter_transcript_support_level(
    gene_annotation.filter_protein_coding(gtf), ["1"]
)

# Extractor for identifying transcripts in a region.
transcript_extractor = transcript.TranscriptExtractor(gtf_transcript)

# Also define an extractor that fetches only the longest transcript per gene.
gtf_longest_transcript = gene_annotation.filter_to_longest_transcript(gtf_transcript)
longest_transcript_extractor = transcript.TranscriptExtractor(gtf_longest_transcript)

## 4. Let's Try it Out: Predict Variant Effects

The MS4A cluster is a very interesting region for AD.  
- There is a strong, replicated GWAS signal, making it one of the top non-APOE loci for late-onset AD. 
- Multiple independent variants show association across cohorts.  
- Signals are non-coding, but the regulatory effects are not fully understood.  
- But...
  - MS4A genes expression is very cell specific, significantly enriched in **microglia** relative to neurons.  
  - **AlphaGenome was not trained on microglia**. 

<img src="./images/bellenguez_manhattan.png" width="100%">


<img src="./images/MS4A_region.png" width="100%">

From Bellenguez et al. 2022 (Image source: [NIAGADS GenomicsDB](https://www.niagads.org/genomics/app/record/track/GCST90027158) / NIAGADS Genome Browser).

The remainder of this workbook will step through two ways AlphaGenome can be used to predict variant effects and lend insight to regulatory impacts of non-coding variants.

In each of the examples, we are going to do the best we can with biological context:

* `brain tissue`: `UBERON:0000955`
* `macrophage`: `CL:0000235` (proxy for `microglia`; both dervive from myleoid lineage, shared TF drivers [SP1, IRFs, etc], similar functional role, some known regulatory overlap)

Example [4A](#4a-predict-variant-effects---gene-expression): Predict the effect of a variant 11:60254475:G:A on a specific output type and tissue by making predictions for the reference (REF) and alternative (ALT) allele sequence.

Example [4B](#4b-scoring-all-possible-snvs-in-a-specific-interval):  Score all possible SNVs in a the MS4A locus to highlight which regions in a DNA sequence are functionally important for variant prediction.

### 4A. Predict variant effects - Gene expression

- Pick a variant e.g., 11:60254475:G:A  (highlighted in the Manhattan plot, above)

Gene expression is primarily captured by the model outputs RNA_SEQ and CAGE. 

Specify the variant (`Variant` object imported from `alphagenome.data.genome`) and define the interval over which to make the `REF/ALT` predictions and the targeted outputs.  The iterval is defined by "resizing" to a model-compatible sequence length.  

> **Resizing** an interval expands (or contracts) the interval centered on the reference location (in this case, the SNV).

In [ ]:
BRAIN = "UBERON:0000955"
MACROPHAGE = "CL:0000235"

variant = Variant(
    chromosome="chr11",
    position=60254475,
    reference_bases="G",  # Can differ from the true reference genome base.
    alternate_bases="A",
)

interval = variant.reference_interval.resize(dna_client.SEQUENCE_LENGTH_1MB)

variant_output = dna_model.predict_variant(
    interval=interval,
    variant=variant,
    requested_outputs=[
        dna_client.OutputType.RNA_SEQ,
        dna_client.OutputType.CAGE,
    ],
    ontology_terms=[MACROPHAGE, BRAIN],
)

print(f"{variant_output}")

Let's plot the results to be able to better inspect & explore.  

Build the plot by constructing multiple tracks (`components`).  

> See AlphaGenome's [visualization guide](https://www.alphagenomedocs.com/visualization_library_basics.html#visualization-basics) for more information.

What does this result tell us?

- If tracks overlap → no predicted effect
- If they diverge → variant likely affects regulation

In [ ]:
transcripts = transcript_extractor.extract(interval=interval)
# Extract the longest transcripts per gene for this interval.
longest_transcripts = longest_transcript_extractor.extract(interval)


plot = plot_components.plot(
    [
        plot_components.OverlaidTracks(
            tdata={
                "REF": variant_output.reference.rna_seq,
                "ALT": variant_output.alternate.rna_seq,
            },
            colors={"REF": "dimgrey", "ALT": "red"},
        ),
        plot_components.TranscriptAnnotation(longest_transcripts),
        plot_components.Tracks(
            tdata=variant_output.reference.rna_seq,
            ylabel_template="RNA_SEQ: {biosample_name} ({strand})\n{name}",
        ),
        plot_components.Tracks(
            tdata=variant_output.reference.cage,
            ylabel_template="CAGE: {biosample_name} ({strand})\n{name}",
        ),
    ],
    # Annotate the location of the variant as a vertical line.
    annotations=[plot_components.VariantAnnotation([variant], alpha=0.8)],
    interval=interval,
    title="Predicted RNA Expression (RNA_SEQ, CAGE) for colon tissue",
)

The answer is - not much of a predicted effect here.  No difference  between the `REF` and `ALT` predictions.  But let's score it anyway. 

Scores per variant are reported in [`AnnData`](https://anndata.readthedocs.io/en/stable/) format.

- `.X` - the scores
- `.obs` - gene metadata
- `.var` - track metadata
- `.uns` - additional unstructured metadata

See the [variant scoring documentation](https://www.alphagenomedocs.com/variant_scoring.html) for more information on variant scorers.

In [ ]:
rna_seq_scorer = variant_scorers.RECOMMENDED_VARIANT_SCORERS[dna_client.OutputType.RNA_SEQ.name]
cage_scorer = variant_scorers.RECOMMENDED_VARIANT_SCORERS[dna_client.OutputType.CAGE.name]

variant_scores = dna_model.score_variant(
    interval=interval, variant=variant, variant_scorers=[rna_seq_scorer, cage_scorer]
)

# e.g., access some of the gene metadata
# we used two scorers, 0 -> RNA SEQ
variant_scores[0].obs.head()

In [ ]:
# ditto for the tracks -> this is the same as the output_metadata, just filtered for the relevant type of output
variant_scores[0].var

Additional, useful metadata in `.uns`:

In [ ]:
print(f'Interval: {variant_scores[0].uns["interval"]}')
print(f'Variant: {variant_scores[0].uns["variant"]}')
print(f'Variant scorer: {variant_scores[0].uns["variant_scorer"]}')

Let's look at the scores.

The `raw_score` column contains the same values as stored in `.X`. The `quantile_score` column is the rank of the `raw_score` in the distribution of scores for a background set of common variants, represented as a quantile probability. More information from the [AlphaGenome FAQ](https://www.alphagenomedocs.com/faqs.html#what-is-the-difference-between-a-quantile-score-and-raw-score).

What are they telling us?

- Raw score (~0.01–0.017) -> very small effect size, i.e., essentially near zero change in expression
- Quantile score (~0.999) -> thje variant ranks in the top ~0.1% of predicted effects genome-wide

> BUT: That ranking is relative, not absolut - even tiny effects can rank high if most variants do nothing.

> The model is predicting consistent but extremely small expression change, which ranks high because most variants do nothing.

> Also note that the predicted impact is on a processed pseudogene.

In [ ]:
# if you want to see the whole table, increase the maxBytes for itables
variant_scorers.tidy_scores([variant_scores[0]], match_gene_strand=True).sort_values("quantile_score", ascending=False)

### 4B. Scoring "all" SNVs in the MS4A locus

Maybe it will be useful to score all the variants in the region to see if we can discover one more likely to have a discernable regulatory effect.

The MS4A locus we are examining here is relative large: 541kb - `chr11:59,914,128-60,455,152` and too large for `ISM` modeling.

> Challenge - [score the interval](https://www.alphagenomedocs.com/api/generated/alphagenome.models.dna_client.DnaClient.html#alphagenome.models.dna_client.DnaClient.score_intervals) - see AlphaGenome Documetation.

In [ ]:
# (optional) score the interval 


Another approach To highlight which regions in the locus are functionally important for a final variant prediction, we can perform an in silico mutagenesis (ISM) analysis by scoring all possible single nucleotide variants in a specific interval. 

> In silico mutagenesis (ISM) systematically mutates each position (or variant) in a sequence and measures the change in a model’s predicted output to quantify the functional importance of that sequence.

Start by definiting the ISM interval.  This analysis is limited to `16KB` range.  Let's pick one of the smaller genes in the locus:

- `MS4A2` - chr11:60,088,261-60,098,467 - ~10.2kb

For this task we will use use a center mask scorer on predicted DNASE values, which will score each variant's effect on DNA accessibility in the 500bp vicinity. 

See the [variant scoring documentation](https://www.alphagenomedocs.com/variant_scoring.html) for more information on variant scorers.

In [ ]:
# define the interval and resize to 16KB
sequence_interval = genome.Interval('chr11', 60_088_261, 60_098_467).resize(dna_client.SEQUENCE_LENGTH_16KB)

# Mutate all bases in the central 256-base region of the sequence_interval.
ism_interval = sequence_interval.resize(256)

# and define the scorer; I'm using DNASE b/c the model was trained on macrophage - DNASE tracks
dnase_variant_scorer = variant_scorers.CenterMaskScorer(
    requested_output=dna_client.OutputType.DNASE,
    width=501,
    aggregation_type=variant_scorers.AggregationType.DIFF_MEAN,
)

# and then score the variants
ism_variant_scores = dna_model.score_ism_variants(
    interval=sequence_interval,
    ism_interval=ism_interval,
    variant_scorers=[dnase_variant_scorer],
)

To identify the most influential positions, the scores can be visualized as a sequence logo, which requires reducing them to a single scalar value per variant.

For example, let's extract the DNASE score just for the brain tissue.  

> Macrophage is there in the scoring output but not mapped to the same ontology term.  Remember how to search the scoring metadata to find out?  Maybe we can get a better result with those or other myleoid like contexts: e.g., `monocyte`

In [ ]:
def extract_macrophage(adata):
  values = adata.X[:, adata.var['ontology_curie'] == BRAIN]
  assert values.size == 1
  return values.flatten()[0]


ism_result = ism.ism_matrix(    
    [extract_macrophage(x[0]) for x in ism_variant_scores],
    variants=[v[0].uns['variant'] for v in ism_variant_scores],
)

And then plot the contribution scores as a sequence logo.  This illustrates how sensistive the model's prediction is to changes at each base position.

- clusters of high-impact positions -> likely to be promoter or enhancer regions
- motif-like patterns -> possible transcription factor binding sites 
- base-level constraints (one letter tall, another near zero) -> changes away from that base cause a large drop in predicted signal so that base is required for maintaining activity

In [ ]:
# Extract transcripts overlapping the ISM interval
transcripts = transcript_extractor.extract(interval=ism_interval)

plot_components.plot(
    [
        plot_components.SeqLogo(
            scores=ism_result,
            scores_interval=ism_interval,
            ylabel='ISM Brain DNase',
        ),
        plot_components.TranscriptAnnotation(transcripts),
    ],
    interval=ism_interval,
    fig_width=10,
)

plt.show()